<a href="https://colab.research.google.com/github/suhailnoor1821/CRM_CW1_Stock_Segmentation/blob/main/CRM_CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.1 Ticker selection and data acquisition
* Tickers: S&P 500 list from Wikipedia (`pd.read_html`), as in Tutorials 04/05.
* Prices: **yfinance** (used in the seminars). `auto_adjust=False` gives both `Adj Close` (for returns and beta) and `Close` (for Monetary); `actions=True` adds dividends and splits for the corporate-action checks.
* Benchmark: `^GSPC`. Period: 1 Jan 2021 to 1 Jan 2025 (end date exclusive).
* **Snapshot:** prices change over time, so the data were downloaded **once on 14 Nov 2026** and saved in the GitHub repository (`data/data_snapshot_2026-11-14`). Every run loads these saved files; the download code only runs if the files are missing.

In [27]:
# Install the yfinance library (used to download share prices)
!pip install -q yfinance

In [28]:
# Import the libraries we need
import os
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Study dates: 1 Jan 2021 to 1 Jan 2025 (the end date itself is not included)
start = '2021-01-01'
end = '2025-01-01'
retrieval_date = '2026-11-14'      # the day we downloaded the data

# Get the saved data (snapshot) from the GitHub repository, so anyone can run this notebook
folder = '/content/CRM_CW1_Stock_Segmentation'
if not os.path.exists(folder):
    !git clone -q https://github.com/suhailnoor1821/CRM_CW1_Stock_Segmentation.git {folder}
snapshot_dir = f'{folder}/data/data_snapshot_{retrieval_date}'
output_dir = f'{folder}/outputs'
os.makedirs(snapshot_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Show the package versions (the brief asks for them)
print('pandas', pd.__version__, '| numpy', np.__version__, '| yfinance', yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [30]:
# Task 1.1 Part 1: Get the list of S&P 500 companies from Wikipedia
# This runs only the first time. After that, the saved list is used
ticker_file = f'{snapshot_dir}/sp500_constituents.csv'

if not os.path.exists(ticker_file):
    # Wikipedia blocks requests without a User-Agent, so we send one
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
    sp500_df = table[0]
    # Yahoo writes BRK.B as BRK-B, so change '.' to '-'
    sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
    sp500_df['Retrieval_Date'] = retrieval_date
    sp500_df.to_csv(ticker_file, index=False)

# Load the saved list
sp500_df = pd.read_csv(ticker_file, keep_default_na=False)
tickers = sp500_df['Ticker'].tolist()

print('Total tickers extracted:', len(tickers), '| retrieved on', retrieval_date)
sp500_df[['Symbol', 'Ticker', 'Security', 'GICS Sector', 'Date added']].head(10)

Total tickers extracted: 503 | retrieved on 2026-11-14


,Symbol,Ticker,Security,GICS Sector,Date added
0,MMM,MMM,3M,Industrials,1957-03-04
1,AOS,AOS,A. O. Smith,Industrials,2017-07-26
2,ABT,ABT,Abbott Laboratories,Health Care,1957-03-04
3,ABBV,ABBV,AbbVie,Health Care,2012-12-31
4,ACN,ACN,Accenture,Information Technology,2011-07-06
5,ADBE,ADBE,Adobe Inc.,Information Technology,1997-05-05
6,AMD,AMD,Advanced Micro Devices,Information Technology,2017-03-20
7,AES,AES,AES Corporation,Utilities,1998-10-02
8,AFL,AFL,Aflac,Financials,1999-05-28
9,A,A,Agilent Technologies,Health Care,2000-06-05


In [31]:
# Task 1.1 Part 2: Daily prices and volume for every share and for the S&P 500 (^GSPC)
# The download runs only if the saved files are missing. After that, the saved files are used
if not os.path.exists(f'{snapshot_dir}/adj_close_daily.csv.gz'):
    df = yf.download(tickers, start=start, end=end, auto_adjust=False, actions=True)
    df['Adj Close'].to_csv(f'{snapshot_dir}/adj_close_daily.csv.gz')
    df['Close'].to_csv(f'{snapshot_dir}/close_daily.csv.gz')
    df['Volume'].to_csv(f'{snapshot_dir}/volume_daily.csv.gz')
    df['Dividends'].to_csv(f'{snapshot_dir}/dividends_daily.csv.gz')
    df['Stock Splits'].to_csv(f'{snapshot_dir}/splits_daily.csv.gz')

    # Download the S&P 500 index (the benchmark)
    market = yf.download('^GSPC', start=start, end=end, auto_adjust=False)
    market.columns = market.columns.get_level_values(0)
    market[['Adj Close', 'Close', 'Volume']].to_csv(f'{snapshot_dir}/gspc_daily.csv.gz')

# Small helper that reads one saved file
def load(name):
    return pd.read_csv(f'{snapshot_dir}/{name}_daily.csv.gz', index_col=0, parse_dates=True)

adj_close = load('adj_close')
close = load('close')
volume = load('volume')
dividends = load('dividends')
splits = load('splits')
gspc = load('gspc')        # S&P 500 index (benchmark)

print('Shares:', adj_close.shape[1], '| trading days:', adj_close.shape[0],
      '| first:', adj_close.index.min().date(), '| last:', adj_close.index.max().date())

Shares: 503 | trading days: 1005 | first: 2021-01-04 | last: 2024-12-31


In [32]:
# Find tickers where Yahoo Finance gave no data at all
no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]
print('Tickers with no data:', len(no_data), no_data)

Tickers with no data: 5 ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
